# LeetCode 2165 — Smallest Value of the Rearranged Number bằng Python

Link đề: https://leetcode.com/problems/smallest-value-of-the-rearranged-number/ · Độ khó: **Medium** · Chủ đề: Math, Sorting

Skeleton LeetCode:

```python
class Solution:
    def smallestNumber(self, num: int) -> int:
        ...
```

Sắp lại các chữ số để số **nhỏ nhất** (không có số 0 đứng đầu, giữ nguyên dấu). Bài ngắn nhưng có **hai cái bẫy**: số âm (nhỏ nhất = trị tuyệt đối **lớn nhất**) và số 0 đứng đầu với số dương. Notebook đi từ **thử mọi hoán vị** → **sắp xếp + đưa chữ số khác 0 nhỏ nhất lên đầu** → **đếm chữ số (counting sort)** → **tham lam từng vị trí** → **thuần số học không dùng chuỗi**.

## 1. Mục tiêu học

1. Phân tích bài theo **dấu**: số dương cần trị tuyệt đối nhỏ nhất, số âm cần trị tuyệt đối **lớn nhất**.
2. Xử lý điều kiện "không có số 0 đứng đầu" bằng cách đưa **chữ số khác 0 nhỏ nhất** lên đầu, rồi các chữ số còn lại tăng dần.
3. Dùng **đếm chữ số** (10 ô) thay cho sắp xếp — `O(số chữ số)`.
4. Chứng minh tính tối ưu bằng **lập luận tham lam theo từng vị trí** (so sánh số cùng độ dài = so sánh từ điển).
5. Kiểm thử đối chiếu với vét cạn hoán vị, và với các trường hợp đặc biệt: `0`, toàn số 0 trừ một chữ số, số âm có số 0.

## 2. Đề bài & ví dụ

Cho số nguyên `num`. Sắp lại các chữ số của `num` để giá trị **nhỏ nhất** và **không có số 0 đứng đầu**. Dấu của số **không đổi**.

**Ví dụ 1:** `num = 310` → `103` (các cách sắp: `013, 031, 103, 130, 301, 310`; bỏ các cách có 0 đầu, nhỏ nhất là `103`).

**Ví dụ 2:** `num = -7605` → `-7650` (một số cách: `-7650, -6705, -5076, -0567`; nhỏ nhất hợp lệ là `-7650`).

**Ràng buộc:** `−10¹⁵ ≤ num ≤ 10¹⁵`.

**Lưu ý:** với số âm, các chữ số sắp **giảm dần** nên số 0 tự nhiên dồn về cuối — không bao giờ có số 0 đứng đầu (trừ khi `num = 0`).

In [1]:
import random
import time
from itertools import permutations
from typing import List

examples = [(310, 103), (-7605, -7650)]

for num, exp in examples:
    digits = sorted(str(abs(num)))
    print(f"num={num}: chữ số {digits}, dấu {'âm' if num < 0 else 'không âm'}, kỳ vọng {exp}")

num=310: chữ số ['0', '1', '3'], dấu không âm, kỳ vọng 103
num=-7605: chữ số ['0', '5', '6', '7'], dấu âm, kỳ vọng -7650


## 3. Kiến thức nền

### 3.1 Tách dấu và trị tuyệt đối

In [2]:
for num in [310, -7605, 0, -10]:
    print(num, "-> âm?", num < 0, "| abs:", abs(num), "| chữ số:", list(str(abs(num))))

310 -> âm? False | abs: 310 | chữ số: ['3', '1', '0']
-7605 -> âm? True | abs: 7605 | chữ số: ['7', '6', '0', '5']
0 -> âm? False | abs: 0 | chữ số: ['0']
-10 -> âm? True | abs: 10 | chữ số: ['1', '0']


### 3.2 So sánh số cùng độ dài = so sánh chữ số từ trái sang

Hai số **cùng số chữ số** (không có 0 đầu): số nào có chữ số đầu tiên khác nhau lớn hơn thì lớn hơn. Đây là cơ sở cho tham lam "chọn chữ số nhỏ nhất có thể ở vị trí đầu, rồi vị trí tiếp theo…".

In [3]:
print("103 < 130 ?", 103 < 130, "| '103' < '130' ?", "103" < "130")

103 < 130 ? True | '103' < '130' ? True


### 3.3 `sorted` trên chuỗi và `str.join`

In [4]:
s = "7605"
print(sorted(s), sorted(s, reverse=True), "".join(sorted(s, reverse=True)))

['0', '5', '6', '7'] ['7', '6', '5', '0'] 7650


### 3.4 Đếm chữ số bằng mảng 10 ô

Có 10 chữ số khả dĩ ⇒ đếm `cnt[d]` rồi xuất theo thứ tự tăng/giảm — counting sort `O(L + 10)`.

In [5]:
cnt = [0] * 10
for ch in "7605":
    cnt[int(ch)] += 1
print("cnt =", cnt, "| xuất tăng dần:", "".join(str(d) * cnt[d] for d in range(10)))

cnt = [1, 0, 0, 0, 0, 1, 1, 1, 0, 0] | xuất tăng dần: 0567


### 3.5 Hai "cạm bẫy" của chuyển đổi chuỗi ↔ số

- `int("013") == 13`: số 0 đầu **biến mất** — nếu ta sắp thành `"013"` rồi đổi sang số, kết quả có **ít chữ số hơn** và không còn là một cách sắp lại hợp lệ.
- `ord("-") = 45 < ord("0") = 48`: khi sắp chuỗi `str(num)` của số âm, dấu trừ luôn bị đưa lên đầu — "tình cờ" đúng vị trí nhưng các chữ số phía sau bị sắp theo chiều sai cho số âm.

In [6]:
print('int("013") =', int("013"), "| số chữ số trước/sau:", len("013"), len(str(int("013"))))
print('ord("-") =', ord("-"), '| ord("0") =', ord("0"), '| sorted("-7605") =', sorted("-7605"))

int("013") = 13 | số chữ số trước/sau: 3 2
ord("-") = 45 | ord("0") = 48 | sorted("-7605") = ['-', '0', '5', '6', '7']


## 4. Ý tưởng nền tảng / trực giác

**Trường hợp `num ≥ 0`.** Mọi cách sắp có **cùng số chữ số** (không được có 0 đầu nên không "rút ngắn" được), nên muốn nhỏ nhất thì chữ số đầu nhỏ nhất có thể — nhưng phải **khác 0**. Sau đó các vị trí còn lại tự do ⇒ xếp tăng dần (số 0 lên trước).

Tính tay `310`: chữ số `{0, 1, 3}`. Đầu: nhỏ nhất khác 0 là `1`. Còn `{0, 3}` tăng dần: `0, 3` ⇒ `103` ✓.

**Trường hợp `num < 0`.** `−x` nhỏ nhất ⇔ `x` **lớn nhất**. Số lớn nhất từ các chữ số: xếp **giảm dần** (chữ số đầu lớn nhất — không thể là 0 trừ khi mọi chữ số là 0, mà khi đó `num = 0`, không âm).

Tính tay `−7605`: chữ số `{7, 6, 0, 5}` giảm dần: `7650` ⇒ `−7650` ✓.

In [7]:
def show_reasoning(num):
    ds = sorted(str(abs(num)))
    if num < 0:
        print(f"  {num}: âm -> xếp giảm {ds[::-1]} -> {-int(''.join(ds[::-1]))}")
    else:
        i = next((k for k, c in enumerate(ds) if c != "0"), None)
        if i is None:
            print(f"  {num}: chỉ có số 0 -> 0"); return
        first = ds[i]; rest = ds[:i] + ds[i + 1:]
        print(f"  {num}: dương -> chữ số đầu = nhỏ nhất khác 0 = {first}, còn lại tăng dần {rest} -> {first + ''.join(rest)}")

for n in [310, -7605, 0, 1000, 5, -40020]:
    show_reasoning(n)

  310: dương -> chữ số đầu = nhỏ nhất khác 0 = 1, còn lại tăng dần ['0', '3'] -> 103
  -7605: âm -> xếp giảm ['7', '6', '5', '0'] -> -7650
  0: chỉ có số 0 -> 0
  1000: dương -> chữ số đầu = nhỏ nhất khác 0 = 1, còn lại tăng dần ['0', '0', '0'] -> 1000
  5: dương -> chữ số đầu = nhỏ nhất khác 0 = 5, còn lại tăng dần [] -> 5
  -40020: âm -> xếp giảm ['4', '2', '0', '0', '0'] -> -42000


## 5. Cách 1 — Brute Force / baseline: thử mọi hoán vị

Sinh mọi hoán vị các chữ số của `|num|`, bỏ hoán vị có 0 đầu (trừ khi số chỉ có một chữ số), gắn dấu, lấy nhỏ nhất. `L ≤ 16` chữ số ⇒ `16!` hoán vị — chỉ dùng được với `L ≤ 8` để đối chiếu.

In [8]:
def smallest_brute(num):
    s = str(abs(num))
    best = None
    for p in set(permutations(s)):
        if len(p) > 1 and p[0] == "0":
            continue
        v = int("".join(p))
        v = -v if num < 0 else v
        if best is None or v < best:
            best = v
    return best

for num, exp in examples:
    print(smallest_brute(num), "(kỳ vọng", exp, ")")

103 (kỳ vọng 103 )
-7650 (kỳ vọng -7650 )


**Độ phức tạp:** `O(L! · L)` — bùng nổ. **Nút thắt:** ta đã biết cấu trúc nghiệm (mục 4), không cần thử.

## 6. Ý tưởng thuật toán / bước đột phá

**Quy tắc:**
- **nếu** `num < 0` **thì** đáp án là `−(chữ số của |num| xếp giảm dần)`;
- **nếu** `num == 0` **thì** đáp án `0`;
- **ngược lại**, xếp tăng dần rồi **đưa chữ số khác 0 đầu tiên lên vị trí 0** (đổi chỗ với số 0 đầu cũng được, vì các số 0 giống hệt nhau).

**Vì sao "đổi chỗ" (swap) cũng đúng?** Sau khi sắp tăng: `0…0 d₁ d₂ …` với `d₁` là chữ số khác 0 nhỏ nhất. Đổi `d₁` với số 0 ở vị trí 0 cho `d₁ 0…0 d₂ …` — chính là "đưa `d₁` lên đầu rồi các số còn lại tăng dần", vì phần còn lại `0…0 d₂…` vẫn tăng dần.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp chuỗi + đổi chỗ chữ số khác 0 đầu tiên

In [9]:
def smallest_sort(num):
    if num < 0:
        return -int("".join(sorted(str(-num), reverse=True)))
    ds = sorted(str(num))
    i = 0
    while i < len(ds) and ds[i] == "0":
        i += 1
    if i < len(ds):
        ds[0], ds[i] = ds[i], ds[0]          # đưa chữ số khác 0 nhỏ nhất lên đầu
    return int("".join(ds))

for num, exp in examples:
    print(smallest_sort(num), "(kỳ vọng", exp, ")")

103 (kỳ vọng 103 )
-7650 (kỳ vọng -7650 )


**Độ phức tạp:** `O(L log L)` với `L ≤ 16` — thực tế hằng số. Khi `num = 0`, vòng `while` chạy hết và không đổi chỗ, `int("0") = 0`.

### 7.2 Cách 3 — Đếm chữ số (counting sort)

In [10]:
def smallest_count(num):
    neg = num < 0
    cnt = [0] * 10
    for ch in str(abs(num)):
        cnt[ord(ch) - 48] += 1
    if neg:
        return -int("".join(str(d) * cnt[d] for d in range(9, -1, -1)))
    first = next((d for d in range(1, 10) if cnt[d]), None)
    if first is None:
        return 0
    cnt[first] -= 1
    return int(str(first) + "".join(str(d) * cnt[d] for d in range(10)))

for num, exp in examples:
    print(smallest_count(num), "(kỳ vọng", exp, ")")

103 (kỳ vọng 103 )
-7650 (kỳ vọng -7650 )


**Độ phức tạp:** `O(L + 10)`, bộ nhớ `O(10)` + chuỗi kết quả.

### 7.3 Cách 4 — Tham lam từng vị trí (dạng tổng quát)

Mô tả lại quy tắc theo kiểu "xây từng chữ số": ở vị trí `k`, chọn chữ số **nhỏ nhất** (với số dương) hoặc **lớn nhất** (với số âm) **còn lại** sao cho vẫn hợp lệ (vị trí 0 của số có nhiều chữ số không được là 0). Viết tổng quát giúp áp dụng cho các biến thể có thêm ràng buộc (mục 15).

In [11]:
def smallest_greedy(num):
    neg = num < 0
    pool = sorted(str(abs(num)), reverse=neg)       # thứ tự ưu tiên chọn
    L = len(pool)
    out = []
    for k in range(L):
        for j, c in enumerate(pool):
            if k == 0 and L > 1 and c == "0":
                continue                              # không cho 0 đứng đầu
            out.append(c); pool.pop(j); break
    v = int("".join(out))
    return -v if neg else v

for num, exp in examples:
    print(smallest_greedy(num), "(kỳ vọng", exp, ")")

103 (kỳ vọng 103 )
-7650 (kỳ vọng -7650 )


**Độ phức tạp:** `O(L²)` — với `L ≤ 16` không đáng kể.

### 7.4 Cách 5 — Thuần số học (không dùng chuỗi)

Lấy chữ số bằng `% 10`, `// 10`; dựng kết quả bằng `res = res * 10 + d`. Cách này dịch thẳng sang C/Java với `long`.

In [12]:
def smallest_arith(num):
    neg = num < 0
    x = -num if neg else num
    cnt = [0] * 10
    if x == 0:
        return 0
    while x:
        cnt[x % 10] += 1
        x //= 10
    res = 0
    if neg:
        for d in range(9, -1, -1):
            for _ in range(cnt[d]):
                res = res * 10 + d
        return -res
    first = next(d for d in range(1, 10) if cnt[d])
    cnt[first] -= 1
    res = first
    for d in range(10):
        for _ in range(cnt[d]):
            res = res * 10 + d
    return res

for num, exp in examples:
    print(smallest_arith(num), "(kỳ vọng", exp, ")")

103 (kỳ vọng 103 )
-7650 (kỳ vọng -7650 )


**Lưu ý tràn số (C/Java):** `|num| ≤ 10¹⁵` có tối đa 16 chữ số; số sắp lại vẫn ≤ `9 999 999 999 999 999 < 2⁶³` ⇒ `long` đủ. Trong Python không cần lo.

## 8. TRACE từng bước

### 8.1 Trace cách 2 trên các ví dụ

In [13]:
def trace_sort(num):
    if num < 0:
        ds = sorted(str(-num), reverse=True)
        print(f"  {num}: âm, xếp giảm {ds} -> {-int(''.join(ds))}")
        return -int("".join(ds))
    ds = sorted(str(num))
    print(f"  {num}: xếp tăng {ds}")
    i = 0
    while i < len(ds) and ds[i] == "0":
        i += 1
    if i < len(ds) and i > 0:
        print(f"    chữ số khác 0 đầu tiên ở vị trí {i} ('{ds[i]}'), đổi với vị trí 0")
        ds[0], ds[i] = ds[i], ds[0]
    print(f"    -> {ds} -> {int(''.join(ds))}")
    return int("".join(ds))

for n in [310, -7605, 100200, 0, 9]:
    _ = trace_sort(n)

  310: xếp tăng ['0', '1', '3']
    chữ số khác 0 đầu tiên ở vị trí 1 ('1'), đổi với vị trí 0
    -> ['1', '0', '3'] -> 103
  -7605: âm, xếp giảm ['7', '6', '5', '0'] -> -7650
  100200: xếp tăng ['0', '0', '0', '0', '1', '2']
    chữ số khác 0 đầu tiên ở vị trí 4 ('1'), đổi với vị trí 0
    -> ['1', '0', '0', '0', '0', '2'] -> 100002
  0: xếp tăng ['0']
    -> ['0'] -> 0
  9: xếp tăng ['9']
    -> ['9'] -> 9


### 8.2 Trace cách 4 (tham lam từng vị trí) trên `5020` (ví dụ tự dựng — đã đối chiếu vét cạn, đối chiếu lại đề gốc)

In [14]:
def trace_greedy(num):
    neg = num < 0
    pool = sorted(str(abs(num)), reverse=neg)
    L = len(pool); out = []
    for k in range(L):
        for j, c in enumerate(pool):
            if k == 0 and L > 1 and c == "0":
                print(f"    vị trí {k}: bỏ qua '0' (không được đứng đầu)")
                continue
            out.append(c); pool.pop(j)
            print(f"    vị trí {k}: chọn '{c}', còn lại {pool}, đã có {''.join(out)}")
            break
    v = int("".join(out)); v = -v if neg else v
    print("  =>", v)
    return v

r = trace_greedy(5020); assert r == smallest_brute(5020)
r = trace_greedy(-5020); assert r == smallest_brute(-5020)

    vị trí 0: bỏ qua '0' (không được đứng đầu)
    vị trí 0: bỏ qua '0' (không được đứng đầu)
    vị trí 0: chọn '2', còn lại ['0', '0', '5'], đã có 2
    vị trí 1: chọn '0', còn lại ['0', '5'], đã có 20
    vị trí 2: chọn '0', còn lại ['5'], đã có 200
    vị trí 3: chọn '5', còn lại [], đã có 2005
  => 2005
    vị trí 0: chọn '5', còn lại ['2', '0', '0'], đã có 5
    vị trí 1: chọn '2', còn lại ['0', '0'], đã có 52
    vị trí 2: chọn '0', còn lại ['0'], đã có 520
    vị trí 3: chọn '0', còn lại [], đã có 5200
  => -5200


### 8.3 Trace cách 3 (đếm chữ số) — bảng `cnt` và chuỗi dựng dần

In [15]:
def trace_count(num):
    neg = num < 0
    cnt = [0] * 10
    for ch in str(abs(num)):
        cnt[int(ch)] += 1
    print(f"  {num}: cnt = {cnt}")
    if neg:
        parts = [(d, str(d) * cnt[d]) for d in range(9, -1, -1) if cnt[d]]
        print("    âm -> ghép d = 9..0:", parts)
        return -int("".join(p for _, p in parts))
    first = next((d for d in range(1, 10) if cnt[d]), None)
    if first is None:
        print("    toàn số 0 -> 0"); return 0
    cnt[first] -= 1
    built = str(first)
    print(f"    chữ số đầu = {first}, cnt còn {cnt}")
    for d in range(10):
        if cnt[d]:
            built += str(d) * cnt[d]
            print(f"    thêm {cnt[d]} chữ số {d}: {built}")
    return int(built)

for n in [4004050, -30020, 7]:
    print("  =>", trace_count(n))

  4004050: cnt = [4, 0, 0, 0, 2, 1, 0, 0, 0, 0]
    chữ số đầu = 4, cnt còn [4, 0, 0, 0, 1, 1, 0, 0, 0, 0]
    thêm 4 chữ số 0: 40000
    thêm 1 chữ số 4: 400004
    thêm 1 chữ số 5: 4000045
  => 4000045
  -30020: cnt = [3, 0, 1, 1, 0, 0, 0, 0, 0, 0]
    âm -> ghép d = 9..0: [(3, '3'), (2, '2'), (0, '000')]
  => -32000
  7: cnt = [0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
    chữ số đầu = 7, cnt còn [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  => 7


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def smallestNumber(self, num: int) -> int:
        if num < 0:
            return -int("".join(sorted(str(-num), reverse=True)))   # âm: trị tuyệt đối lớn nhất
        cnt = [0] * 10
        for ch in str(num):
            cnt[ord(ch) - 48] += 1
        for d in range(1, 10):                                      # chữ số khác 0 nhỏ nhất lên đầu
            if cnt[d]:
                cnt[d] -= 1
                return int(str(d) + "".join(str(x) * cnt[x] for x in range(10)))
        return 0                                                    # num == 0

for num, exp in examples:
    assert Solution().smallestNumber(num) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**`num > 0`.** Mọi cách sắp hợp lệ có đúng `L` chữ số (chữ số đầu khác 0), nên so sánh giá trị = so sánh từ điển của chuỗi `L` ký tự.
- Vị trí 0: phải khác 0; nhỏ nhất có thể là `d₁` = chữ số khác 0 nhỏ nhất. Mọi cách sắp có chữ số đầu `> d₁` đều lớn hơn mọi cách sắp bắt đầu bằng `d₁`.
- Cố định `d₁`, phần còn lại là `L − 1` chữ số **không có ràng buộc** (số 0 được phép); chuỗi nhỏ nhất theo từ điển của một đa tập chữ số là chuỗi **tăng dần** (đổi chỗ bất kỳ cặp nghịch thế nào cũng làm chuỗi nhỏ đi).
⇒ `d₁` + (phần còn lại tăng dần) là tối ưu.

**`num < 0`.** `−x < −y ⇔ x > y`. Muốn `x` lớn nhất: chuỗi giảm dần (lập luận đối xứng). Chữ số đầu là chữ số lớn nhất, khác 0 vì `num ≠ 0` ⇒ không vi phạm "không có số 0 đầu".

**`num = 0`.** Chỉ có một cách sắp: `0`.

**Đếm chữ số** (cách 3, code nộp) chỉ là cài đặt sắp xếp tăng/giảm bằng counting sort.

### 10.1 Minh họa lập luận trao đổi (exchange argument)

Lấy một cách sắp hợp lệ **bất kỳ** của số dương. Nếu phần đuôi (từ vị trí 1) **chưa tăng dần**, tồn tại cặp kề `a[i] > a[i+1]` với `i ≥ 1`; đổi chỗ cặp này cho một số **nhỏ hơn** mà vẫn hợp lệ (vị trí 0 không đổi). Nếu chữ số đầu chưa phải chữ số khác 0 nhỏ nhất `d₁`, đổi nó với một `d₁` ở đuôi cũng làm số nhỏ hơn. Lặp lại đến khi không cải thiện được ⇒ đúng dạng nghiệm của ta. Cell dưới thực hiện "leo dốc" này từ một cách sắp ngẫu nhiên và luôn dừng ở đáp án:

In [17]:
def improve_until_stuck(digits):
    a = list(digits); steps = 0
    changed = True
    while changed:
        changed = False
        d1 = min(c for c in a if c != "0")
        if a[0] != d1:
            j = a.index(d1, 1); a[0], a[j] = a[j], a[0]; changed = True; steps += 1; continue
        for i in range(1, len(a) - 1):
            if a[i] > a[i + 1]:
                a[i], a[i + 1] = a[i + 1], a[i]; changed = True; steps += 1; break
    return "".join(a), steps

rnd_e = random.Random(10)
for _ in range(500):
    n = rnd_e.randint(1, 10**9)
    ds = list(str(n)); rnd_e.shuffle(ds)
    if ds[0] == "0":
        k = next(i for i, c in enumerate(ds) if c != "0"); ds[0], ds[k] = ds[k], ds[0]
    res, _ = improve_until_stuck(ds)
    assert int(res) == Solution().smallestNumber(n)
start = list("530201")
print("từ", "".join(start), "->", improve_until_stuck(start), "| đáp án:", Solution().smallestNumber(530201))

từ 530201 -> ('100235', 5) | đáp án: 100235


## 11. Phân tích Time / Space Complexity

Gọi `L` = số chữ số của `|num|` (`L ≤ 16`).

| Cách | Time | Space |
|---|---|---|
| 1. Mọi hoán vị | `O(L! · L)` | `O(L!)` (set) |
| 2. Sắp + đổi chỗ | `O(L log L)` | `O(L)` |
| 3. Đếm chữ số | `O(L + 10)` | `O(10 + L)` |
| 4. Tham lam từng vị trí | `O(L²)` | `O(L)` |
| 5. Số học | `O(L + 10)` | `O(10)` |

In [18]:
rnd = random.Random(2165)
qs = [rnd.randint(-10**15, 10**15) for _ in range(30000)]
for name, fn in [("sort", smallest_sort), ("count", smallest_count), ("greedy", smallest_greedy),
                 ("arith", smallest_arith), ("Solution", Solution().smallestNumber)]:
    t0 = time.perf_counter()
    for q in qs:
        fn(q)
    print(f"{name:<9}: 30000 số trong {(time.perf_counter() - t0) * 1000:7.1f} ms")

sort     : 30000 số trong    36.0 ms


count    : 30000 số trong    59.5 ms


greedy   : 30000 số trong    77.7 ms


arith    : 30000 số trong    62.0 ms


Solution : 30000 số trong    46.2 ms


## 12. Bộ test

In [19]:
ALL = {
    "sort": smallest_sort,
    "count": smallest_count,
    "greedy": smallest_greedy,
    "arith": smallest_arith,
    "Solution": lambda n: Solution().smallestNumber(n),
}

def is_valid(num, out):
    """Validator: cùng dấu, cùng đa tập chữ số, không có 0 đầu."""
    if (num < 0) != (out < 0):
        return False
    a, b = str(abs(num)), str(abs(out))
    return sorted(a) == sorted(b) and (len(b) == 1 or b[0] != "0")

def check(num, exp=None, brute=True):
    ref = smallest_brute(num) if brute else smallest_sort(num)
    assert is_valid(num, ref)
    for name, fn in ALL.items():
        got = fn(num)
        assert got == ref, (name, num, got, ref)
    if exp is not None:
        assert ref == exp, (num, ref, exp)
    return ref

for num, exp in examples:
    check(num, exp)
print("Ví dụ đề: OK")

check(0, 0)
check(5, 5); check(-5, -5)
check(10, 10); check(-10, -10)
check(1000, 1000); check(-1000, -1000)
check(100200, 100002)
check(-40020, -42000)
check(987654, 456789)
check(10**15, 10**15, brute=False)
check(-10**15, -10**15, brute=False)
check(-123456789012345, -987655443322110, brute=False)
print("Edge case: OK")

rnd = random.Random(2021)
for _ in range(1500):
    L = rnd.randint(1, 7)
    digits = [rnd.choice("0001234569") for _ in range(L)]
    if L > 1 and digits[0] == "0":
        digits[0] = rnd.choice("123456789")
    n = int("".join(digits)) * rnd.choice([1, -1])
    check(n)
print("1500 số ngẫu nhiên ≤ 7 chữ số (nhiều số 0) so với vét cạn: OK")

for _ in range(3000):
    n = rnd.randint(-10**15, 10**15)
    r = check(n, brute=False)
    assert is_valid(n, r)
print("3000 số ngẫu nhiên trong miền ràng buộc (đối chiếu chéo + validator): OK")

Ví dụ đề: OK
Edge case: OK


1500 số ngẫu nhiên ≤ 7 chữ số (nhiều số 0) so với vét cạn: OK
3000 số ngẫu nhiên trong miền ràng buộc (đối chiếu chéo + validator): OK


In [20]:
# Vét cạn TOÀN BỘ các số từ -9999 đến 9999 (mọi số ≤ 4 chữ số)
for n in range(-9999, 10000):
    assert Solution().smallestNumber(n) == smallest_brute(n), n
print("Mọi số trong [-9999, 9999] khớp vét cạn hoán vị: OK")

Mọi số trong [-9999, 9999] khớp vét cạn hoán vị: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Sắp tăng dần rồi `int(...)` (số 0 đầu bị "nuốt")

In [21]:
def bug_plain_sort(num):
    s = sorted(str(abs(num)))
    v = int("".join(s))
    return -v if num < 0 else v

print("310 -> sai:", bug_plain_sort(310), "| đúng:", Solution().smallestNumber(310))

310 -> sai: 13 | đúng: 103


**Vì sao sai:** `"013"` → `int` = `13`, **mất một chữ số** — không phải một cách sắp lại hợp lệ của `310`. **Sửa:** đưa chữ số khác 0 nhỏ nhất lên đầu.

### Lỗi 2 — Số âm cũng sắp tăng dần

In [22]:
def bug_negative_ascending(num):
    s = sorted(str(abs(num)))
    i = next((k for k, c in enumerate(s) if c != "0"), 0)
    s[0], s[i] = s[i], s[0]
    v = int("".join(s))
    return -v if num < 0 else v

print("-7605 -> sai:", bug_negative_ascending(-7605), "| đúng:", Solution().smallestNumber(-7605))

-7605 -> sai: -5067 | đúng: -7650


**Vì sao sai:** `−5067 > −7650`. Với số âm, "nhỏ nhất" nghĩa là trị tuyệt đối **lớn nhất**. **Sửa:** sắp giảm dần khi `num < 0`.

### Lỗi 3 — Sắp chuỗi `str(num)` có cả dấu `-`

In [23]:
def bug_sort_with_sign(num):
    s = sorted(str(num))                       # '-' có mã ASCII 45 < '0' (48) nên đứng đầu
    return int("".join(s))

print("-7605 -> sai:", bug_sort_with_sign(-7605), "| đúng:", Solution().smallestNumber(-7605))

-7605 -> sai: -567 | đúng: -7650


**Vì sao sai:** dấu được sắp như một ký tự; kết quả `−0567 = −567` mất chữ số và sai thứ tự. **Sửa:** tách dấu, làm việc với `abs(num)`.

### Lỗi 4 — Dồn mọi số 0 ra **sau** chữ số khác 0 cuối cùng

In [24]:
def bug_zeros_at_end(num):
    if num < 0:
        return -int("".join(sorted(str(-num), reverse=True)))
    s = str(num)
    nz = sorted(c for c in s if c != "0")
    return int("".join(nz) + "0" * s.count("0")) if nz else 0

print("310 -> sai:", bug_zeros_at_end(310), "| đúng:", Solution().smallestNumber(310))
print("100200 -> sai:", bug_zeros_at_end(100200), "| đúng:", Solution().smallestNumber(100200))

310 -> sai: 130 | đúng: 103
100200 -> sai: 120000 | đúng: 100002


**Vì sao sai:** chỉ vị trí **đầu tiên** cần khác 0; các số 0 nên đứng **ngay sau** chữ số đầu (càng sớm càng tốt) để số nhỏ nhất. **Sửa:** `d₁ + "0"·z + (các chữ số khác 0 còn lại tăng dần)`.

### Lỗi 5 — Chọn chữ số đầu là chữ số **lớn nhất** khác 0 cho số dương (nhầm hướng với số âm)

In [25]:
def bug_first_max(num):
    if num <= 0:
        return Solution().smallestNumber(num)
    s = sorted(str(num))
    first = max(c for c in s if c != "0")
    s.remove(first)
    return int(first + "".join(s))

print("310 -> sai:", bug_first_max(310), "| đúng:", Solution().smallestNumber(310))

310 -> sai: 301 | đúng: 103


**Sửa:** số dương: chữ số đầu là chữ số khác 0 **nhỏ nhất**. Bài học chung cho cả 5 lỗi: viết validator (`is_valid`) + vét cạn hoán vị cho số ngắn sẽ bắt được hết.

### Lỗi 6 — Làm việc với `abs(num)` rồi quên gắn lại dấu

In [26]:
def bug_forget_sign(num):
    x = abs(num)
    if num < 0:
        return int("".join(sorted(str(x), reverse=True)))     # SAI: thiếu dấu trừ
    return Solution().smallestNumber(x)

print("-7605 -> sai:", bug_forget_sign(-7605), "| đúng:", Solution().smallestNumber(-7605))

-7605 -> sai: 7650 | đúng: -7650


**Sửa:** `return -int(...)`. Validator `is_valid` ở mục 12 kiểm **dấu** trước tiên chính để bắt lỗi này.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài "sắp lại chữ số để nhỏ/lớn nhất" ⇒ **tách dấu trước**, đảo chiều mục tiêu cho số âm.
- "Không có số 0 đầu" ⇒ chỉ ràng buộc **vị trí đầu tiên**; phần còn lại tự do.
- Số chữ số rất nhỏ (≤ 16) ⇒ sắp xếp hay đếm đều `O(1)` trong thực tế; chọn cách dễ đọc nhất.
- Python một dòng cho số dương: `s = sorted(str(num)); i = next(...)` rồi swap.
- Test luôn gồm: `0`, số một chữ số, số có nhiều số 0, số âm có số 0, biên `±10¹⁵`.

## 15. Biến thể của bài

1. **Số lớn nhất** sau khi sắp lại (dương: giảm dần; âm: đưa chữ số khác 0 nhỏ nhất lên đầu rồi tăng dần) — bài tập 3.
2. **LeetCode 670 — Maximum Swap:** chỉ được **một** lần đổi chỗ để có số lớn nhất.
3. **LeetCode 31 — Next Permutation:** hoán vị kế tiếp theo thứ tự từ điển.
4. **LeetCode 179 — Largest Number:** ghép các **số** thành số lớn nhất (so sánh `a+b` vs `b+a`).
5. **Số nhỏ nhất chia hết cho 3 / chẵn** từ các chữ số — tham lam từng vị trí có kiểm tra khả thi (bài tập 5 dùng ràng buộc "chẵn").

In [27]:
def maximum_swap(num: int) -> int:
    """LeetCode 670: tối đa một lần đổi chỗ để có số lớn nhất."""
    s = list(str(num))
    last = {c: i for i, c in enumerate(s)}
    for i, c in enumerate(s):
        for d in "9876543210":
            if d > c and last.get(d, -1) > i:
                j = last[d]; s[i], s[j] = s[j], s[i]
                return int("".join(s))
    return num

def maximum_swap_brute(num):
    s = list(str(num)); best = num
    for i in range(len(s)):
        for j in range(i + 1, len(s)):
            t = s[:]; t[i], t[j] = t[j], t[i]
            best = max(best, int("".join(t)))
    return best

rnd_v = random.Random(15)
for _ in range(2000):
    n = rnd_v.randint(0, 10**8)
    assert maximum_swap(n) == maximum_swap_brute(n)
print("Maximum Swap 2736 ->", maximum_swap(2736), "| 9973 ->", maximum_swap(9973), "| khớp vét cạn: OK")

Maximum Swap 2736 -> 7236 | 9973 -> 9973 | khớp vét cạn: OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi hoán vị | thử hết | `O(L! L)` | `O(L!)` | kiểm thử `L ≤ 8` |
| 2. Sắp + đổi chỗ | tăng dần, swap số 0 đầu | `O(L log L)` | `O(L)` | ngắn gọn |
| 3. Đếm chữ số | counting sort | `O(L + 10)` | `O(10)` | **nộp bài** |
| 4. Tham lam từng vị trí | chọn tốt nhất còn hợp lệ | `O(L²)` | `O(L)` | biến thể có ràng buộc |
| 5. Số học | `% 10`, `*10 +` | `O(L + 10)` | `O(10)` | C/Java |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().smallestNumber(-30020)` và `Solution().smallestNumber(30020)`?

**Bài 2 (tính tay):** Với `num = 4004050`, liệt kê `cnt` và dựng đáp án theo cách 3.

**Bài 3 (điền TODO):** Hoàn thiện `largest_number(num)` — sắp lại chữ số để số **lớn nhất** (không có 0 đầu, giữ dấu).

**Bài 4 (tự viết lại):** Viết lại cách 2 **không dùng `sorted`**, chỉ dùng mảng đếm và vòng lặp; trả về chuỗi rồi `int`.

**Bài 5 (mini challenge):** `smallest_even(num)` với `num > 0`: số **chẵn** nhỏ nhất sắp lại được từ các chữ số (không 0 đầu), hoặc `-1` nếu không có. Gợi ý: chọn chữ số chẵn đặt cuối, phần trước xếp như bài gốc; thử mọi lựa chọn chữ số chẵn cuối. Kiểm bằng vét cạn.

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def largest_number(num):
    # TODO: dương -> giảm dần; âm -> như bài gốc với |num| (chữ số khác 0 nhỏ nhất lên đầu, tăng dần)
    pass

# Bài 4 — TODO
def smallest_no_sorted(num):
    # TODO: cnt[10], dựng chuỗi theo d = 0..9 hoặc 9..0
    pass

# Bài 5 — TODO
def smallest_even(num):
    # TODO: với mỗi chữ số chẵn e có trong num: bỏ một e, dựng số nhỏ nhất từ phần còn lại (không 0 đầu), nối e
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `−30020`: âm ⇒ giảm dần.
- Mức 2: `32000` ⇒ `−32000`.
- Mức 3: `30020`: đầu `2`, còn `0,0,0,3` ⇒ `20003`.

**Bài 2**
- Mức 1: chữ số `4,0,0,4,0,5,0`.
- Mức 2: `cnt[0] = 4, cnt[4] = 2, cnt[5] = 1`.
- Mức 3: đầu `4`, còn `0000 4 5` ⇒ `4000045`.

**Bài 3**
- Mức 1: Đối xứng với bài gốc.
- Mức 2: Số âm lớn nhất ⇔ trị tuyệt đối nhỏ nhất (có ràng buộc 0 đầu).
- Mức 3: `largest_number(n) == -smallestNumber(-n)`?  Kiểm bằng vét cạn.

**Bài 4**
- Mức 1: `cnt[ord(c) - 48] += 1`.
- Mức 2: Số dương: tìm `d ≥ 1` nhỏ nhất có `cnt[d] > 0`.
- Mức 3: Ghép `str(d) * cnt[d]`.

**Bài 5**
- Mức 1: Số chẵn ⇔ chữ số cuối chẵn.
- Mức 2: Phần đầu (nếu còn ≥ 2 chữ số) phải không có 0 đầu; nếu chỉ còn toàn số 0 và dài ≥ 1 ⇒ không hợp lệ (trừ khi số chỉ có 1 chữ số).
- Mức 3: Lấy min trên các lựa chọn `e`; so vét cạn hoán vị.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1
r1 = (Solution().smallestNumber(-30020), Solution().smallestNumber(30020))
print("Bài 1:", r1); assert r1 == (-32000, 20003) == (smallest_brute(-30020), smallest_brute(30020))

# Bài 2
cnt = [0] * 10
for ch in "4004050":
    cnt[int(ch)] += 1
r2 = Solution().smallestNumber(4004050)
print("Bài 2: cnt =", cnt, "->", r2); assert r2 == 4000045 == smallest_brute(4004050)

# Bài 3
def largest_number_sol(num):
    if num >= 0:
        return int("".join(sorted(str(num), reverse=True)))
    return -Solution().smallestNumber(-num)

def largest_brute(num):
    s = str(abs(num)); best = None
    for p in set(permutations(s)):
        if len(p) > 1 and p[0] == "0":
            continue
        v = int("".join(p)); v = -v if num < 0 else v
        best = v if best is None or v > best else best
    return best

rnd = random.Random(3)
for _ in range(1500):
    n = rnd.randint(-10**5, 10**5)
    assert largest_number_sol(n) == largest_brute(n)
print("Bài 3 OK, ví dụ:", largest_number_sol(310), largest_number_sol(-7605))

Bài 1: (-32000, 20003)
Bài 2: cnt = [4, 0, 0, 0, 2, 1, 0, 0, 0, 0] -> 4000045
Bài 3 OK, ví dụ: 310 -5067


In [30]:
# Bài 4
def smallest_no_sorted_sol(num):
    neg = num < 0
    cnt = [0] * 10
    for ch in str(abs(num)):
        cnt[ord(ch) - 48] += 1
    if neg:
        return -int("".join(str(d) * cnt[d] for d in range(9, -1, -1)))
    for d in range(1, 10):
        if cnt[d]:
            cnt[d] -= 1
            return int(str(d) + "".join(str(x) * cnt[x] for x in range(10)))
    return 0

# Bài 5
def smallest_even_sol(num):
    s = str(num); best = None
    for e in set(c for c in s if int(c) % 2 == 0):
        rest = list(s); rest.remove(e)
        if not rest:
            cand = int(e)
        else:
            rest.sort()
            i = next((k for k, c in enumerate(rest) if c != "0"), None)
            if i is None:
                continue                      # phần đầu toàn số 0 -> có 0 đứng đầu
            rest[0], rest[i] = rest[i], rest[0]
            cand = int("".join(rest) + e)
        best = cand if best is None or cand < best else best
    return -1 if best is None else best

def smallest_even_brute(num):
    best = None
    for p in set(permutations(str(num))):
        if (len(p) > 1 and p[0] == "0") or int(p[-1]) % 2:
            continue
        v = int("".join(p)); best = v if best is None or v < best else best
    return -1 if best is None else best

rnd = random.Random(5)
for _ in range(1500):
    n = rnd.randint(-10**5, 10**5)
    assert smallest_no_sorted_sol(n) == smallest_brute(n)
    m = rnd.randint(1, 10**5)
    assert smallest_even_sol(m) == smallest_even_brute(m)
print("Bài 4, Bài 5 OK; ví dụ smallest_even:", smallest_even_sol(310), smallest_even_sol(135), smallest_even_sol(4321), smallest_even_sol(100))

Bài 4, Bài 5 OK; ví dụ smallest_even: 130 -1 1234 100


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
if num < 0:
    return -int("".join(sorted(str(-num), reverse=True)))
s = sorted(str(num))
i = next((k for k, c in enumerate(s) if c != "0"), None)
if i is None: return 0
s[0], s[i] = s[i], s[0]
return int("".join(s))
```

**Checklist:**
- [ ] Tách dấu, làm với `abs(num)`?
- [ ] Số âm: **giảm dần**?
- [ ] Số dương: chữ số khác 0 **nhỏ nhất** lên đầu, số 0 ngay sau?
- [ ] `num = 0` trả `0`?
- [ ] Không để `int()` "nuốt" số 0 đầu (mất chữ số)?
- [ ] Chữ ký đúng: `def smallestNumber(self, num: int) -> int`?